# Reinforcement Learning Intro Walkthrough

This notebook is the interactive companion to `gymnasium_rl_experiment.py`. Use it to inspect the Gymnasium interaction loop one cell at a time, compare a random baseline with a trained policy, and read the saved experiment artifacts.

For repeatable homework runs, fixed command-line arguments, smoke checks, and saved result directories, use `gymnasium_rl_experiment.py` directly.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Import the Chapter Runner

The notebook reuses the chapter runner instead of duplicating the experiment logic. It works when opened from this directory or from the repository root.

In [ ]:
from __future__ import annotations

import csv
import json
import platform
import sys
from argparse import Namespace
from pathlib import Path


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError("Open this notebook from the chapter directory or the repository root.")


CHAPTER_CODE_DIR = find_chapter_dir("gymnasium_rl_experiment.py", "chapter_reinforcement_learning_intro")

if str(CHAPTER_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CHAPTER_CODE_DIR))

import gymnasium_rl_experiment as rl_experiment

print("Chapter code:", CHAPTER_CODE_DIR)
print("Python:", platform.python_version())

## 2. Check the RL Dependencies

A full run needs Gymnasium, Stable-Baselines3, PyTorch, NumPy, and Box2D for the default LunarLander environment. From the shared repository environment, install the locked RL dependency group with:

```sh
poetry install --with rl-intro
```

In [ ]:
dependency_report = rl_experiment.dependency_report()
print(json.dumps(dependency_report, indent=2, sort_keys=True))

missing_required = [
    name
    for name, status in dependency_report["required"].items()
    if not status["available"]
]
if missing_required:
    raise RuntimeError(
        "Missing required packages: "
        + ", ".join(missing_required)
        + ". Install from the repository root with: poetry install --with rl-intro."
    )

## 3. Set the Experiment Constants

Keep the environment, algorithm, seed, training budget, and evaluation rule visible. Changing these values changes the evidence you can report.

In [ ]:
ENV_ID = "LunarLander-v3"
ALGORITHM = "ppo"
SEED = 7
TRAIN_STEPS = 100_000
EVAL_EPISODES = 20
N_ENVS = 4
SUCCESS_RETURN_THRESHOLD = 200
OUTPUT_DIR = CHAPTER_CODE_DIR / "runs" / "notebook-lunarlander-ppo"

MODEL_KWARGS = {
    "n_steps": 1024,
    "batch_size": 64,
    "n_epochs": 4,
}

print("Environment:", ENV_ID)
print("Algorithm:", ALGORITHM)
print("Seed:", SEED)
print("Training steps:", TRAIN_STEPS)
print("Evaluation episodes:", EVAL_EPISODES)
print("Output directory:", OUTPUT_DIR)

## 4. Inspect One Environment Step

Before training, inspect the observation space, action space, starting observation, sampled action, reward, and termination flags.

In [ ]:
env = rl_experiment.make_env(ENV_ID, seed=SEED)
try:
    observation, info = env.reset(seed=SEED)
    action = env.action_space.sample()
    next_observation, reward, terminated, truncated, step_info = env.step(action)

    print("Observation space:", env.observation_space)
    print("Action space:", env.action_space)
    print("Initial observation:", observation)
    print("Sampled action:", action)
    print("Reward:", reward)
    print("Terminated:", terminated)
    print("Truncated:", truncated)
    print("Next observation:", next_observation)
finally:
    env.close()

## 5. Run a Short Random Rollout

A random rollout makes the interaction loop concrete: action, reward, termination flag, and truncation flag at each step.

In [ ]:
def show_rows(rows: list[dict], limit: int = 10) -> None:
    try:
        from IPython.display import display

        display(rows[:limit])
    except Exception:
        for row in rows[:limit]:
            print(row)


env = rl_experiment.make_env(ENV_ID, seed=SEED + 1)
try:
    random_episode = rl_experiment.run_episode(
        env=env,
        policy="random",
        seed=SEED + 1,
        max_steps=20,
        deterministic=True,
        success_return_threshold=SUCCESS_RETURN_THRESHOLD,
    )
finally:
    env.close()

print("Random rollout return:", random_episode["return"])
print("Random rollout steps:", random_episode["steps"])
show_rows(random_episode["rollout"], limit=20)

## 6. Evaluate the Random Baseline

The homework comparison should start with a fixed random-policy baseline under the same evaluation episode count used for the trained policy.

In [ ]:
random_summary, random_rows, random_rollout = rl_experiment.evaluate_policy(
    env_id=ENV_ID,
    policy="random",
    episodes=EVAL_EPISODES,
    seed=SEED + 1000,
    max_steps=None,
    deterministic=True,
    success_return_threshold=SUCCESS_RETURN_THRESHOLD,
    env_wrapper="none",
)

print(json.dumps(random_summary, indent=2, sort_keys=True))
show_rows(random_rows, limit=5)

## 7. Train and Evaluate One Agent

This cell calls the same `run` function used by the command-line script. It writes `summary.json`, `config.json`, `evaluation_episodes.csv`, rollout CSVs, and Stable-Baselines3 monitor logs.

In [ ]:
def make_run_args(output_dir: Path, train_steps: int, model_kwargs: dict) -> Namespace:
    return Namespace(
        env_id=ENV_ID,
        algorithm=ALGORITHM,
        policy="MlpPolicy",
        env_wrapper="none",
        train_steps=train_steps,
        eval_episodes=EVAL_EPISODES,
        seed=SEED,
        n_envs=N_ENVS,
        learning_rate=0.0003,
        model_kwargs=json.dumps(model_kwargs),
        max_episode_steps=None,
        success_return_threshold=SUCCESS_RETURN_THRESHOLD,
        device="auto",
        deterministic_eval=True,
        output_dir=output_dir,
    )


summary = rl_experiment.run(make_run_args(OUTPUT_DIR, TRAIN_STEPS, MODEL_KWARGS))

print("Random baseline mean return:", summary["random_baseline"]["mean_return"])
print("Trained policy mean return:", summary["trained_policy"]["mean_return"])
print("Trained policy success rate:", summary["trained_policy"]["success_rate"])
print("Training seconds:", round(summary["training"]["training_seconds"], 3))
print("Saved summary:", OUTPUT_DIR / "summary.json")

## 8. Load the Saved Artifacts

The saved files are the evidence for the report. The notebook reads them back instead of relying only on variables still in memory.

In [ ]:
summary_path = OUTPUT_DIR / "summary.json"
evaluation_path = OUTPUT_DIR / "evaluation_episodes.csv"
random_rollout_path = OUTPUT_DIR / "random_rollout.csv"
trained_rollout_path = OUTPUT_DIR / "trained_rollout.csv"

loaded_summary = json.loads(summary_path.read_text(encoding="utf-8"))

with evaluation_path.open("r", encoding="utf-8", newline="") as handle:
    evaluation_rows = list(csv.DictReader(handle))

print("Artifacts:")
for name, path in loaded_summary["artifacts"].items():
    print(f"- {name}: {path}")

show_rows(evaluation_rows, limit=8)

## 9. Compare Random and Trained Returns

A plot is useful for reading the evaluation episodes, but the JSON and CSV files remain the source of record. If Plotnine is unavailable, the cell prints grouped returns instead.

In [ ]:
returns_by_policy = {}
for row in evaluation_rows:
    returns_by_policy.setdefault(row["policy"], []).append(float(row["return"]))

try:
    import pandas as pd
    from plotnine import aes, geom_hline, geom_line, geom_point, ggplot, labs, theme_minimal

    records = []
    for policy, returns in returns_by_policy.items():
        for episode, return_value in enumerate(returns, start=1):
            records.append({"policy": policy, "episode": episode, "return_value": return_value})
    plot = (
        ggplot(pd.DataFrame.from_records(records), aes("episode", "return_value", color="policy", group="policy"))
        + geom_line(size=0.8)
        + geom_point(size=2.0)
        + geom_hline(yintercept=SUCCESS_RETURN_THRESHOLD, color="gray", linetype="dashed", size=0.6)
        + labs(title=f"{ENV_ID} evaluation returns", x="Evaluation episode", y="Return", color="policy")
        + theme_minimal()
    )
    plot
except ImportError:
    print("Install Plotnine, or install the shared figures group, to draw the plot.")
    print(json.dumps(returns_by_policy, indent=2, sort_keys=True))


## 10. Inspect Qualitative Behavior

The rollout CSVs are a small qualitative artifact. They show whether the trained policy lasts longer and how the reward sequence differs from the random policy.

In [ ]:
def load_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


random_rollout_rows = load_csv(random_rollout_path)
trained_rollout_rows = load_csv(trained_rollout_path)

print("Random rollout length:", len(random_rollout_rows))
print("Trained rollout length:", len(trained_rollout_rows))
print("First trained rollout steps:")
show_rows(trained_rollout_rows, limit=10)

## 11. Run One Optional Ablation

A useful ablation changes exactly one factor. This example changes PPO `clip_range` while keeping the environment, seed, training steps, and evaluation rule fixed. Set `RUN_ABLATION` to `True` when you want to run it.

In [ ]:
RUN_ABLATION = False
ABLATION_MODEL_KWARGS = {
    **MODEL_KWARGS,
    "clip_range": 0.1,
}
ABLATION_OUTPUT_DIR = CHAPTER_CODE_DIR / "runs" / "notebook-lunarlander-ppo-clip-0.1"

if RUN_ABLATION:
    ablation_summary = rl_experiment.run(make_run_args(ABLATION_OUTPUT_DIR, TRAIN_STEPS, ABLATION_MODEL_KWARGS))
    print("Main trained mean return:", summary["trained_policy"]["mean_return"])
    print("Ablation trained mean return:", ablation_summary["trained_policy"]["mean_return"])
    print("Changed factor: PPO clip_range from default to 0.1")
else:
    print("Set RUN_ABLATION = True to run the one-factor ablation.")

## 12. Fill the Report Fields

Use the saved summary and evaluation artifacts to fill the chapter report template.

In [ ]:
report_fields = {
    "Environment": loaded_summary["environment"],
    "Algorithm": loaded_summary["algorithm"],
    "Random baseline mean return": round(loaded_summary["random_baseline"]["mean_return"], 3),
    "Trained policy mean return": round(loaded_summary["trained_policy"]["mean_return"], 3),
    "Success rate or score threshold": loaded_summary["trained_policy"]["success_rate"],
    "Total environment steps": loaded_summary["train_steps"],
    "Training time and hardware": (
        f"{loaded_summary['training']['training_seconds']:.2f}s on "
        f"{loaded_summary['training']['device']} ({loaded_summary['runtime']['platform']})"
    ),
    "Ablation changed factor": "Run the optional ablation cell and record the changed factor.",
    "Qualitative behavior": f"Trained rollout lasted {len(trained_rollout_rows)} steps.",
}

for field, value in report_fields.items():
    print(f"{field}: {value}")

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.